In [ ]:
# pipeline_local_models.py

import pandas as pd
import torch
import math
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import gc



bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

max_memory = {
    0: "6GB",   # GPU
    "cpu": "12GB"
}

# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     device_map="auto",
#     quantization_config=bnb_config,
# )

token = "hf_tok"


local_models = [
    # Hugging Face model ID, compatible with small GPU
    "TheBloke/vicuna-7B-1.1-HF",         # Quantized Vicuna 7B
    # "NousResearch/Nous-Hermes-7B",       # Efficient 7B instruct model
    # "TheBloke/WizardLM-7B-uncensored",
    # "mistralai/Mistral-7B-Instruct-v0.1"
    # "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

   # 7B instruction-tuned
]

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")





def get_logprobs(sentence, tokenizer, model):
    """
    Returns per-token log-probs (as token, logprob tuples),
    surprisal, avg log-prob, and overall probability.
    """
    inputs = tokenizer(sentence, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # Shift logits and labels
    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]

    log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)
    token_logprobs_tensor = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)

    # Convert to list
    token_logprobs = token_logprobs_tensor[0].cpu().tolist()

    # Convert token IDs to readable tokens
    token_ids = shift_labels[0].cpu().tolist()
    tokens = tokenizer.convert_ids_to_tokens(token_ids)

    # Zip into (token, logprob)
    token_logprobs = list(zip(tokens, token_logprobs))

    # Aggregate stats
    total_log_prob = sum(lp for _, lp in token_logprobs)
    avg_log_prob = total_log_prob / len(token_logprobs)
    surprisal = -avg_log_prob
    overall_prob = math.exp(total_log_prob)

    return surprisal, avg_log_prob, overall_prob, token_logprobs


def analyze_sentences_df(df, model_id):
    print(f"Loading model {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=token,
        device_map="auto",
        offload_folder="./offload",
        offload_state_dict=True,          # Automatically map layers to GPU/CPU
        quantization_config=bnb_config,
        low_cpu_mem_usage=True,   # FP16 for GPU efficiency
        max_memory=max_memory,

    )

    results = []
    for idx, row in df.iterrows():
        sentence = row['sentence']
        

        surprisal, avg_log, prob, tokens = get_logprobs(sentence, tokenizer, model)

        results.append({

            "surprisal": surprisal,
            "avg_log_prob": avg_log,
            "acceptability_prob": prob,
            "log_probs_per_token": tokens,           
            
        })
   
    return pd.DataFrame(results)
    



c:\Users\Devin\Documents\ABE LLM LAB\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [4]:

if __name__ == "__main__":
    # Load your dataset
    df = pd.read_csv("exp3_ST.csv")

    # Run all local models
    for model_name in local_models:
        # # 1. Delete specific objects
        # del model
        # del optimizer
        # del tensor

        # 2. Invoke garbage collector
        gc.collect()

        # 3. Clear GPU cache
        torch.cuda.empty_cache()
        print(f"\n=== Analyzing with model: {model_name} ===")
        results_df = analyze_sentences_df(df, model_name)
        results_df=pd.concat((df, results_df), axis=1)
        results_df.to_csv(f"results_exp3{model_name.replace('/', '_')}.csv", index=False)
        print(f"Results saved for {model_name}")
        


=== Analyzing with model: TheBloke/vicuna-7B-1.1-HF ===
Loading model TheBloke/vicuna-7B-1.1-HF...


Loading weights: 100%|██████████| 291/291 [00:14<00:00, 19.92it/s]


Results saved for TheBloke/vicuna-7B-1.1-HF
